In [ ]:
import pandas as pd
import numpy as np

session_df = pd.read_csv("../data/analysis_table/session_table.csv")
user_df = pd.read_csv("../data/analysis_table/user_table.csv")
funnel = pd.read_csv("../data/analysis_table/funnel_table.csv")
event_df = pd.read_csv("../data/analysis_table/2019_Oct_cleaned.csv")

In [ ]:
# define "Cart Abandonment": when a session has a cart event but didn't have purchase event.
abandoned = (session_df['cart_count'] > 0) & (session_df['purchase_count'] == 0)

total_sessions = len(session_df)

cart_sessions = (session_df['cart_count'] > 0).sum()
purchase_sessions = (session_df['purchase_count'] > 0).sum()
abandoned_sessions = abandoned.sum()

# calculate Cart-to-purchase rate and Abandonment rate
cart_to_purchase_rate = purchase_sessions / cart_sessions
abandonment_rate = abandoned_sessions / cart_sessions

print(cart_to_purchase_rate)

In [ ]:
# First of all, only taking the "view" events, since users see the prices in "view" events.
view_df = event_df[event_df['event_type'] == 'view'].copy()

session_price = (view_df.groupby('user_session')
                 .agg(session_avg_price = ('price', 'mean'))
                 .reset_index()
                 )

session_df = session_df.merge(session_price, on = 'user_session', how = 'left')

bins = [0, 50, 100, 500, 1000, 2000]
labels = ['0-50', '50-100', '100-500', '500-1000', '1000+']

session_df['price_bucket'] = pd.cut(session_df['session_avg_price'], bins = bins, labels = labels)

session_df[session_df['has_purchase'] > 0].head()

In [ ]:
price_analysis = (session_df.groupby('price_bucket')
                  .agg(sessions = ('user_session', 'count'),
                       cart_rate = ('has_cart', 'mean'),
                       purchase_rate = ('has_purchase', 'mean')
                  ).reset_index()
                )

print(price_analysis)